# 12 · The KV Cache and the Memory Wall

Companion to **Chapter 15**. You will verify the cache is *exactly* correct,
measure the speedup, and do the arithmetic that governs every serving decision.

In [ ]:
import os, sys, time, math
import torch
sys.path.insert(0, os.path.abspath('..'))
from tfs.model import Config, Model

torch.manual_seed(0)
cfg = Config(vocab_size=512, d_model=128, n_layer=4, n_head=4,
             n_kv_head=2, d_head=32, d_ff=340, max_T=512)
model = Model(cfg).eval()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 1 · Correctness first

A fast cache that changes the output is worthless. Two tests.

In [ ]:
# TEST A -- feeding one token at a time with a cache == feeding all at once
ids = torch.randint(0, cfg.vocab_size, (1, 12))
with torch.no_grad():
    full, _, _ = model(ids)                       # (1, 12, V) in one pass

    caches, outs = None, []
    for t in range(ids.size(1)):
        lg, _, caches = model(ids[:, t:t+1], caches=caches, pos_offset=t)
        outs.append(lg[:, -1])
    incremental = torch.stack(outs, dim=1)        # (1, 12, V)

print(f"max abs diff: {(full - incremental).abs().max().item():.2e}")
assert torch.allclose(full, incremental, atol=1e-4)
print("incremental == full forward ✓")

# TEST B -- greedy generation must be TOKEN-IDENTICAL with and without cache
prompt = torch.randint(0, cfg.vocab_size, (1, 8))
a = model.generate(prompt.clone(), 40, temperature=0, use_cache=False)
b = model.generate(prompt.clone(), 40, temperature=0, use_cache=True)
print(f"cached == uncached: {torch.equal(a, b)} ✓")
assert torch.equal(a, b)

### The three bugs this test catches

Break each one deliberately and watch the assert fail.

In [ ]:
# BUG 1: wrong RoPE position offset during decode
with torch.no_grad():
    caches, outs = None, []
    for t in range(ids.size(1)):
        lg, _, caches = model(ids[:, t:t+1], caches=caches, pos_offset=0)  # <- always 0
        outs.append(lg[:, -1])
    bad = torch.stack(outs, dim=1)

print(f"pos_offset=0 always -> max diff {(full - bad).abs().max().item():.4f}")
assert not torch.allclose(full, bad, atol=1e-4)
print("  every token thinks it is at position 0. Coherent for a few tokens, then drifts.")
print("\n(Bug 2 -- is_causal=True during decode -- is prevented in tfs.model by")
print(" `is_causal=(T > 1)`. With one query row, a causal mask aligns TOP-LEFT,")
print(" so the single query would see only key 0 and ignore the whole prompt.)")

## 2 · The speedup

Uncached generation is O(T²) in total work; cached is O(T).

In [ ]:
def time_generate(n_new, use_cache):
    p = torch.randint(0, cfg.vocab_size, (1, 8))
    t0 = time.perf_counter()
    model.generate(p, n_new, temperature=0, use_cache=use_cache)
    return time.perf_counter() - t0

print(f"{'tokens':>8} {'no cache':>11} {'cache':>10} {'speedup':>9}")
for n in [25, 50, 100, 200]:
    t_no = time_generate(n, False)
    t_yes = time_generate(n, True)
    print(f"{n:>8} {t_no:>10.3f}s {t_yes:>9.3f}s {t_no/t_yes:>8.1f}x")

print("\nThe speedup GROWS with length: uncached reprocesses the whole prefix")
print("every step, so its cumulative cost is quadratic.")

## 3 · Cache size — the number that decides your GPU bill

In [ ]:
def kv_bytes(n_layer, n_kv_head, d_head, seq, batch=1, dtype_bytes=2):
    """2 (K and V) x layers x kv_heads x head_dim x seq x batch x bytes"""
    return 2 * n_layer * n_kv_head * d_head * seq * batch * dtype_bytes

def fmt(b):
    for u in ["B","KB","MB","GB","TB"]:
        if b < 1024: return f"{b:7.2f} {u}"
        b /= 1024
    return f"{b:.1f} PB"

models = {
    #                 layers  kv_heads  d_head
    "Llama-3-8B  GQA-8":  (32,  8, 128),
    "Llama-3-70B GQA-8":  (80,  8, 128),
    "70B if it were MHA": (80, 64, 128),
}
print(f"{'model':>22} {'per token':>11} {'8k ctx':>11} {'128k ctx':>11}")
for name, (L, KV, D) in models.items():
    per = kv_bytes(L, KV, D, 1)
    print(f"{name:>22} {fmt(per):>11} {fmt(per*8192):>11} {fmt(per*131072):>11}")

print("\nWithout GQA the 70B model's cache at 128k would be 320 GB for ONE user --")
print("more than 4x H100 can hold alongside the 140 GB of weights. GQA is not an")
print("optimisation; it is what makes long-context serving possible at all.")

## 4 · Exercise 13.2 — the serving calculator

In [ ]:
def plan(n_params, n_layer, n_kv_head, d_head, gpu_gb, n_gpu,
         context, bandwidth_tbs, w_bytes=2, kv_bytes_per=2, overhead_gb=20):
    weights = n_params * w_bytes / 1024**3
    total = gpu_gb * n_gpu
    avail = total - weights - overhead_gb
    per_user = kv_bytes(n_layer, n_kv_head, d_head, context,
                        dtype_bytes=kv_bytes_per) / 1024**3
    return {
        "weights_gb": weights,
        "cache_budget_gb": avail,
        "gb_per_user": per_user,
        "max_users": max(0, int(avail / per_user)) if per_user else 0,
        "batch1_tok_s": bandwidth_tbs * 1024 / (weights + per_user),
    }

print("Llama-3-70B on 8x H100 (80 GB, 3.35 TB/s), 32k context:\n")
for label, kvb in [("bf16 KV", 2), ("fp8  KV", 1)]:
    r = plan(70e9, 80, 8, 128, 80, 8, 32768, 3.35, kv_bytes_per=kvb)
    print(f"  {label}:  weights {r['weights_gb']:.0f} GB   "
          f"budget {r['cache_budget_gb']:.0f} GB   "
          f"{r['gb_per_user']:.2f} GB/user   "
          f"-> {r['max_users']} concurrent users")

print("\nfp8 KV cache roughly doubles your concurrency for a small accuracy cost.")
print("That is why it is standard in production.")

## 5 · Why decode is memory-bound (Exercise 13.3)

The single most important calculation in LLM inference.

In [ ]:
H100_TFLOPS, H100_TBS = 990, 3.35
ridge = H100_TFLOPS * 1e12 / (H100_TBS * 1e12)
print(f"H100 ridge point: {ridge:.0f} FLOP per byte")
print("  (above this you are compute-bound; below it, memory-bound)\n")

print(f"{'batch':>7} {'intensity':>11} {'regime':>16}")
for bs in [1, 8, 32, 128, 256, 512]:
    intensity = bs        # ~1 FLOP/byte per sequence in bf16
    regime = "memory-bound" if intensity < ridge else "compute-bound"
    print(f"{bs:>7} {intensity:>11} {regime:>16}")

print(f"\nYou need batch ~{ridge:.0f} to saturate an H100 during decode.")
print("That is why inference providers batch aggressively -- and why your API")
print("latency is often BETTER when the service is busy than when it is idle.\n")

print("Batch-1 decode ceiling (bandwidth / bytes-per-token):")
for name, params in [("7B", 7e9), ("13B", 13e9), ("70B", 70e9)]:
    gb = params * 2 / 1024**3
    print(f"  {name:>4} bf16: {gb:6.1f} GB -> {H100_TBS*1024/gb:6.0f} tok/s ceiling")
print("\nReal systems hit 40-60% of this. The estimate is usually within 2x --")
print("remarkable for a one-line model of a very complex system.")

---
## Self-check

1. Why can K and V be cached but not the attention weights?
2. Why is prefill compute-bound and decode memory-bound, with the same weights?
3. During decode with `T=1`, why must `is_causal` be `False`?
4. 40 GB cache budget, 128 KB/token, 8k context per user — how many users?

<details><summary>Answers</summary>

1. Causality: token *t*'s K and V depend only on tokens ≤ *t*, so they never change.
   Attention weights depend on the *current* query, which is new every step.
2. Arithmetic intensity. Prefill reuses each loaded weight `T` times; decode uses it
   once. Same bytes moved, `T`× the work.
3. With one query row a causal mask aligns to the top-left, treating that query as
   position 0 — so it could only see key 0 and would ignore the entire prompt.
4. 8192 × 128 KB = 1.05 GB/user → ~38. Fragmentation cuts this further, which is
   what PagedAttention fixes (Chapter 18).

</details>

**Next:** `16_flashattention.ipynb`